# Job Queue with SSE Status Page Example - Cross-Tab Synchronized

> Demonstrates a real-time job queue management system with cancellable background jobs, live progress tracking via Server-Sent Events (SSE), granular UI updates using HTMX SSE extension with out-of-band swaps, and **cross-tab synchronization** that keeps all browser tabs in sync.

## Key Features:
- **Cross-Tab Synchronization**: All browser tabs stay synchronized via global SSE broadcasting
- **Job Running Check**: Prevents multiple concurrent jobs unless explicitly allowed
- **Client-Side Protection**: Double-submit protection on form submission
- **Real-time Updates**: Live progress bars and status badges via SSE
- **Connection Status Indicator**: Shows real-time connection status
- **Automatic Reconnection**: Handles tab visibility changes and reconnects SSE when needed
- **Broadcast Architecture**: Server broadcasts updates to all connected clients when actions occur

In [1]:
from fasthtml.common import *
from fasthtml.common import sse_message
import uuid, time, threading
from datetime import datetime
from typing import Dict, Any
import asyncio
import random
import json

from cjm_tqdm_capture.progress_monitor import ProgressMonitor
from cjm_tqdm_capture.job_runner import JobRunner
from cjm_tqdm_capture.streaming import sse_stream_async

In [2]:
# For Jupyter display
from fasthtml.jupyter import JupyUvi, HTMX
from cjm_fasthtml_daisyui.core.testing import create_test_app, create_test_page, start_test_server
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from IPython.display import display

# Import DaisyUI factories
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors, btn_sizes, btn_behaviors
from cjm_fasthtml_daisyui.components.actions.modal import modal, modal_box, modal_action
from cjm_fasthtml_daisyui.components.feedback.progress import progress, progress_colors
from cjm_fasthtml_daisyui.components.feedback.alert import alert, alert_colors
from cjm_fasthtml_daisyui.components.data_display.card import card
from cjm_fasthtml_daisyui.components.data_display.table import table, table_modifiers
from cjm_fasthtml_daisyui.components.data_display.badge import badge, badge_colors, badge_sizes
from cjm_fasthtml_daisyui.components.data_display.stat import stat, stat_title, stat_value, stats
from cjm_fasthtml_daisyui.components.data_display.status import status, status_colors, status_sizes
from cjm_fasthtml_daisyui.components.data_input.text_input import text_input
from cjm_fasthtml_daisyui.components.data_input.select import select
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, text_dui

# Import Tailwind factories
from cjm_fasthtml_tailwind.utilities.spacing import p, m, space
from cjm_fasthtml_tailwind.utilities.typography import font_weight, font_size, text_color
from cjm_fasthtml_tailwind.utilities.sizing import w, max_w, max_h, container
from cjm_fasthtml_tailwind.utilities.layout import overflow, display_tw
from cjm_fasthtml_tailwind.utilities.borders import rounded
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import flex_display, gap
from cjm_fasthtml_tailwind.utilities.effects import shadow
from cjm_fasthtml_tailwind.core.base import combine_classes

In [3]:
# Core SSE Broadcast Manager abstraction
import asyncio
from typing import Set, Dict, Any, Optional, Callable, Deque
from collections import deque
from datetime import datetime
import json

class SSEBroadcastManager:
    """
    Manages SSE connections and broadcasting without UI dependencies.
    
    This class provides a reusable abstraction for managing Server-Sent Events
    connections and broadcasting messages to multiple clients.
    """
    
    def __init__(self, 
                 max_queue_size: int = 100,
                 history_size: int = 50,
                 default_timeout: float = 0.1):
        """
        Initialize the SSE Broadcast Manager.
        
        Args:
            max_queue_size: Maximum size for client message queues
            history_size: Number of broadcast messages to keep in history
            default_timeout: Default timeout for sending messages to clients
        """
        self.connections: Set[asyncio.Queue] = set()
        self.lock = asyncio.Lock()
        self.history: Deque[Dict[str, Any]] = deque(maxlen=history_size)
        self.max_queue_size = max_queue_size
        self.default_timeout = default_timeout
        
        # Event hooks for extensibility
        self._on_connect_hooks: list[Callable] = []
        self._on_disconnect_hooks: list[Callable] = []
        self._on_broadcast_hooks: list[Callable] = []
    
    async def register_connection(self, queue: Optional[asyncio.Queue] = None) -> asyncio.Queue:
        """
        Register a new SSE connection.
        
        Args:
            queue: Optional pre-existing queue, creates new one if not provided
            
        Returns:
            The queue associated with this connection
        """
        if queue is None:
            queue = asyncio.Queue(maxsize=self.max_queue_size)
        
        async with self.lock:
            self.connections.add(queue)
            
        # Call connection hooks
        for hook in self._on_connect_hooks:
            try:
                await hook(queue) if asyncio.iscoroutinefunction(hook) else hook(queue)
            except Exception as e:
                print(f"Error in connection hook: {e}")
        
        return queue
    
    async def unregister_connection(self, queue: asyncio.Queue):
        """
        Unregister an SSE connection.
        
        Args:
            queue: The queue to unregister
        """
        async with self.lock:
            self.connections.discard(queue)
        
        # Call disconnection hooks
        for hook in self._on_disconnect_hooks:
            try:
                await hook(queue) if asyncio.iscoroutinefunction(hook) else hook(queue)
            except Exception as e:
                print(f"Error in disconnection hook: {e}")
    
    async def broadcast(self, 
                       event_type: str, 
                       data: Dict[str, Any],
                       timeout: Optional[float] = None) -> int:
        """
        Broadcast a message to all connected clients.
        
        Args:
            event_type: Type of event being broadcast
            data: Data to broadcast
            timeout: Optional timeout override for this broadcast
            
        Returns:
            Number of successfully notified connections
        """
        message = {
            "type": event_type,
            "timestamp": datetime.now().isoformat(),
            "data": data
        }
        
        # Store in history
        self.history.append(message)
        
        # Call broadcast hooks
        for hook in self._on_broadcast_hooks:
            try:
                result = await hook(message) if asyncio.iscoroutinefunction(hook) else hook(message)
                # Allow hooks to modify the message
                if isinstance(result, dict):
                    message = result
            except Exception as e:
                print(f"Error in broadcast hook: {e}")
        
        # Broadcast to all connections
        timeout = timeout or self.default_timeout
        successful_sends = 0
        
        async with self.lock:
            disconnected = set()
            
            for queue in self.connections:
                try:
                    await asyncio.wait_for(queue.put(message), timeout=timeout)
                    successful_sends += 1
                except (asyncio.TimeoutError, asyncio.QueueFull):
                    disconnected.add(queue)
                except Exception as e:
                    print(f"Error broadcasting to connection: {e}")
                    disconnected.add(queue)
            
            # Remove disconnected clients
            for queue in disconnected:
                self.connections.discard(queue)
        
        return successful_sends
    
    def on_connect(self, callback: Callable):
        """Register a callback for new connections."""
        self._on_connect_hooks.append(callback)
        return callback
    
    def on_disconnect(self, callback: Callable):
        """Register a callback for disconnections."""
        self._on_disconnect_hooks.append(callback)
        return callback
    
    def on_broadcast(self, callback: Callable):
        """Register a callback for broadcasts (can modify messages)."""
        self._on_broadcast_hooks.append(callback)
        return callback
    
    @property
    def connection_count(self) -> int:
        """Get the current number of active connections."""
        return len(self.connections)
    
    def get_history(self, limit: Optional[int] = None) -> list[Dict[str, Any]]:
        """
        Get broadcast history.
        
        Args:
            limit: Optional limit on number of messages to return
            
        Returns:
            List of historical broadcast messages
        """
        if limit:
            return list(self.history)[-limit:]
        return list(self.history)

In [4]:
# SSE Element Updater abstraction
from typing import Callable, Dict, Any, List, Optional, Union

class SSEElementUpdater:
    """
    Builds OOB swap elements without hardcoding UI components.
    This class provides a flexible system for registering and executing
    element update handlers based on event types.
    """
    
    def __init__(self):
        """Initialize the SSE Element Updater."""
        self._handlers: Dict[str, List[Callable]] = {}
        self._default_handler: Optional[Callable] = None
        self._preprocessors: List[Callable] = []
        self._postprocessors: List[Callable] = []
    
    def register(self, event_type: str, priority: int = 0):
        """
        Decorator to register an update handler for a specific event type.
        
        Args:
            event_type: The event type to handle
            priority: Handler priority (higher numbers run first)
            
        Returns:
            Decorator function
        """
        def decorator(handler: Callable):
            if event_type not in self._handlers:
                self._handlers[event_type] = []
            
            # Add handler with priority
            self._handlers[event_type].append((priority, handler))
            # Sort by priority (descending)
            self._handlers[event_type].sort(key=lambda x: x[0], reverse=True)
            
            return handler
        return decorator
    
    def register_handler(self, event_type: str, handler: Callable, priority: int = 0):
        """
        Register an update handler programmatically.
        
        Args:
            event_type: The event type to handle
            handler: The handler function
            priority: Handler priority (higher numbers run first)
        """
        if event_type not in self._handlers:
            self._handlers[event_type] = []
        
        self._handlers[event_type].append((priority, handler))
        self._handlers[event_type].sort(key=lambda x: x[0], reverse=True)
    
    def set_default_handler(self, handler: Callable):
        """
        Set a default handler for unregistered event types.
        
        Args:
            handler: The default handler function
        """
        self._default_handler = handler
    
    def add_preprocessor(self, processor: Callable):
        """
        Add a preprocessor that runs before handlers.
        
        Args:
            processor: Function that processes (event_type, data) and returns modified data
        """
        self._preprocessors.append(processor)
    
    def add_postprocessor(self, processor: Callable):
        """
        Add a postprocessor that runs after handlers.
        
        Args:
            processor: Function that processes elements list and returns modified elements
        """
        self._postprocessors.append(processor)
    
    def create_elements(self, event_type: str, data: Dict[str, Any]) -> List[Any]:
        """
        Create elements for a given event type and data.
        
        Args:
            event_type: The type of event
            data: Event data
            
        Returns:
            List of elements to be sent via SSE
        """
        # Run preprocessors
        processed_data = data
        for processor in self._preprocessors:
            processed_data = processor(event_type, processed_data)
        
        elements = []
        
        # Get handlers for this event type
        handlers = self._handlers.get(event_type, [])
        
        if handlers:
            # Run all registered handlers
            for priority, handler in handlers:
                result = handler(processed_data)
                if result:
                    if isinstance(result, list):
                        elements.extend(result)
                    else:
                        elements.append(result)
        elif self._default_handler:
            # Use default handler if no specific handlers registered
            result = self._default_handler(event_type, processed_data)
            if result:
                if isinstance(result, list):
                    elements.extend(result)
                else:
                    elements.append(result)
        
        # Run postprocessors
        for processor in self._postprocessors:
            elements = processor(elements)
        
        return elements
    
    def clear_handlers(self, event_type: Optional[str] = None):
        """
        Clear handlers for a specific event type or all handlers.
        
        Args:
            event_type: Optional specific event type to clear
        """
        if event_type:
            self._handlers.pop(event_type, None)
        else:
            self._handlers.clear()
    
    def get_registered_events(self) -> List[str]:
        """Get list of registered event types."""
        return list(self._handlers.keys())


# Helper function for creating OOB swap elements
def oob_swap(element, swap_type: str = "outerHTML", target_id: Optional[str] = None):
    """
    Add OOB swap attributes to an element.
    
    Args:
        element: The element to add OOB swap to
        swap_type: Type of swap (innerHTML, outerHTML, beforebegin, afterend, etc.)
        target_id: Optional target ID (uses element's ID if not specified)
        
    Returns:
        The element with OOB swap attributes
    """
    if hasattr(element, 'attrs'):
        element.attrs['hx-swap-oob'] = swap_type
        if target_id:
            element.attrs['hx-swap-oob'] = f"{swap_type}:#{target_id}"
    return element

def oob_element(element_id: str, content, swap_type: str = "innerHTML"):
    """
    Create a wrapper element for OOB swap.
    
    Args:
        element_id: ID of the target element
        content: Content to swap
        swap_type: Type of swap
        
    Returns:
        Div with OOB swap configuration
    """
    return Div(
        content,
        id=element_id,
        hx_swap_oob=swap_type
    )

In [5]:
# HTMX SSE Connector helpers
from typing import Optional, Dict, Any, Union

class HTMXSSEConnector:
    """
    Provides helper functions for setting up HTMX SSE connections
    without hardcoding specific implementations.
    """
    
    @staticmethod
    def add_sse_attrs(element, 
                      endpoint: str,
                      events: Optional[Union[str, List[str]]] = None,
                      swap_type: str = "message",
                      auto_reconnect: bool = True):
        """
        Add SSE connection attributes to an element.
        
        Args:
            element: The element to add SSE attributes to
            endpoint: SSE endpoint URL
            events: Optional event name(s) to listen for
            swap_type: How to swap content (message, innerHTML, outerHTML, etc.)
            auto_reconnect: Whether to auto-reconnect on disconnect
            
        Returns:
            The element with SSE attributes added
        """
        if hasattr(element, 'attrs'):
            element.attrs['hx-ext'] = 'sse'
            element.attrs['sse-connect'] = endpoint
            
            if events:
                if isinstance(events, list):
                    for event in events:
                        element.attrs[f'sse-swap'] = event
                else:
                    element.attrs['sse-swap'] = events
            else:
                element.attrs['sse-swap'] = swap_type
                
            if not auto_reconnect:
                element.attrs['sse-close'] = 'true'
        
        return element
    
    @staticmethod
    def create_sse_element(element_type=Div,
                          endpoint: str = None,
                          element_id: str = None,
                          events: Optional[Union[str, List[str]]] = None,
                          swap_type: str = "message",
                          hidden: bool = False,
                          **kwargs):
        """
        Create an element with SSE connection configured.
        
        Args:
            element_type: Type of element to create (Div, Span, etc.)
            endpoint: SSE endpoint URL
            element_id: Optional element ID
            events: Optional event name(s) to listen for
            swap_type: How to swap content
            hidden: Whether to hide the element
            **kwargs: Additional attributes for the element
            
        Returns:
            Element configured for SSE connection
        """
        attrs = kwargs.copy()
        
        if element_id:
            attrs['id'] = element_id
            
        if endpoint:
            attrs['hx_ext'] = 'sse'
            attrs['sse_connect'] = endpoint
            
            if events:
                if isinstance(events, list):
                    # For multiple events, would need special handling
                    attrs['sse_swap'] = events[0] if events else swap_type
                else:
                    attrs['sse_swap'] = events
            else:
                attrs['sse_swap'] = swap_type
        
        if hidden:
            attrs['style'] = attrs.get('style', '') + 'display: none;'
        
        return element_type(**attrs)
    
    @staticmethod
    def sse_progress_element(job_id: str, 
                            endpoint_template: str = "/stream_job_progress?job_id={job_id}",
                            element_id_template: str = "progress-span-{job_id}",
                            initial_content=None):
        """
        Create an SSE-enabled progress element.
        
        Args:
            job_id: Job identifier
            endpoint_template: Template for SSE endpoint URL
            element_id_template: Template for element ID
            initial_content: Initial content to display
            
        Returns:
            SSE-configured element for progress updates
        """
        return Span(
            initial_content or "",
            id=element_id_template.format(job_id=job_id),
            hx_ext="sse",
            sse_connect=endpoint_template.format(job_id=job_id),
            sse_swap="message"
        )
    
    @staticmethod
    def sse_status_element(job_id: str,
                          endpoint_template: str = "/stream_job_status?job_id={job_id}",
                          element_id_template: str = "status-span-{job_id}",
                          initial_content=None):
        """
        Create an SSE-enabled status element.
        
        Args:
            job_id: Job identifier
            endpoint_template: Template for SSE endpoint URL
            element_id_template: Template for element ID
            initial_content: Initial content to display
            
        Returns:
            SSE-configured element for status updates
        """
        return Span(
            initial_content or "",
            id=element_id_template.format(job_id=job_id),
            hx_ext="sse",
            sse_connect=endpoint_template.format(job_id=job_id),
            sse_swap="message"
        )
    
    @staticmethod
    def create_sse_monitor_script(config: Dict[str, Any]):
        """
        Create a monitoring script for SSE connections.
        
        Args:
            config: Configuration dictionary with keys:
                - sse_element_id: ID of SSE element to monitor
                - status_element_id: ID of status display element
                - auto_reconnect: Whether to auto-reconnect
                - debug: Whether to enable debug logging
                - status_indicators: Dict of status HTML strings
                
        Returns:
            Script element with monitoring code
        """
        sse_id = config.get('sse_element_id', 'sse-connection')
        status_id = config.get('status_element_id', 'connection-status')
        auto_reconnect = config.get('auto_reconnect', True)
        debug = config.get('debug', False)
        indicators = config.get('status_indicators', {})
        
        # Default indicators if not provided
        if not indicators:
            indicators = {
                'active': '<span style="color: green;">Connected</span>',
                'disconnected': '<span style="color: red;">Disconnected</span>',
                'error': '<span style="color: orange;">Error</span>',
                'reconnecting': '<span style="color: blue;">Reconnecting...</span>'
            }
        
        js_code = f"""
        (function() {{
            const sseElementId = '{sse_id}';
            const statusElementId = '{status_id}';
            const debugLogging = {str(debug).lower()};
            const autoReconnect = {str(auto_reconnect).lower()};
            
            const statusIndicators = {json.dumps(indicators)};
            
            function log(message) {{
                if (debugLogging) {{
                    console.log('[SSE Monitor] ' + message);
                }}
            }}
            
            function updateStatus(status) {{
                const statusElement = document.getElementById(statusElementId);
                if (statusElement && statusIndicators[status]) {{
                    statusElement.innerHTML = statusIndicators[status];
                    log('Status updated to: ' + status);
                }}
            }}
            
            document.addEventListener('DOMContentLoaded', function() {{
                const sseElement = document.getElementById(sseElementId);
                
                if (!sseElement) {{
                    log('SSE element not found: ' + sseElementId);
                    return;
                }}
                
                // Monitor SSE connection events
                sseElement.addEventListener('htmx:sseOpen', function(evt) {{
                    log('SSE connection established');
                    updateStatus('active');
                }});
                
                sseElement.addEventListener('htmx:sseClose', function(evt) {{
                    log('SSE connection closed');
                    updateStatus('disconnected');
                }});
                
                sseElement.addEventListener('htmx:sseError', function(evt) {{
                    log('SSE connection error');
                    updateStatus('error');
                    
                    if (autoReconnect) {{
                        setTimeout(function() {{
                            updateStatus('reconnecting');
                            htmx.trigger(sseElement, 'htmx:sseReconnect');
                        }}, 3000);
                    }}
                }});
                
                // Handle tab visibility changes
                if (autoReconnect) {{
                    document.addEventListener('visibilitychange', function() {{
                        if (!document.hidden) {{
                            const sseData = sseElement['htmx-internal-data'];
                            
                            if (sseData && sseData.sseEventSource) {{
                                const state = sseData.sseEventSource.readyState;
                                
                                if (state === EventSource.CLOSED) {{
                                    log('Tab visible - reconnecting closed SSE');
                                    updateStatus('reconnecting');
                                    htmx.trigger(sseElement, 'htmx:sseReconnect');
                                }} else if (state === EventSource.OPEN) {{
                                    log('Tab visible - SSE already connected');
                                    updateStatus('active');
                                }} else if (state === EventSource.CONNECTING) {{
                                    log('Tab visible - SSE connecting');
                                    updateStatus('reconnecting');
                                }}
                            }}
                        }} else {{
                            log('Tab hidden');
                        }}
                    }});
                }}
                
                // Initial status check
                setTimeout(function() {{
                    const sseData = sseElement['htmx-internal-data'];
                    if (sseData && sseData.sseEventSource) {{
                        const state = sseData.sseEventSource.readyState;
                        if (state === EventSource.OPEN) {{
                            updateStatus('active');
                        }} else if (state === EventSource.CONNECTING) {{
                            updateStatus('reconnecting');
                        }} else {{
                            updateStatus('disconnected');
                        }}
                    }}
                }}, 100);
            }});
        }})();
        """
        
        return Script(js_code)

# Initialize the HTMX SSE Connector
htmx_sse = HTMXSSEConnector()

In [6]:
# SSE Event Dispatcher for decoupled event routing
from typing import Callable, Dict, Any, List, Optional, Set
from dataclasses import dataclass
import re

@dataclass
class SSEEvent:
    """Represents an SSE event with metadata."""
    type: str
    data: Dict[str, Any]
    namespace: Optional[str] = None
    priority: int = 0
    timestamp: Optional[str] = None
    
    @property
    def full_type(self):
        """Get the full event type including namespace."""
        if self.namespace:
            return f"{self.namespace}:{self.type}"
        return self.type

class SSEEventDispatcher:
    """
    Decoupled event routing system with namespace support,
    middleware, filtering, and priority-based handling.
    """
    
    def __init__(self):
        """Initialize the SSE Event Dispatcher."""
        self._handlers: Dict[str, List[tuple[int, Callable]]] = {}
        self._middleware: List[Callable] = []
        self._filters: List[Callable] = []
        self._transformers: List[Callable] = []
        self._namespaces: Set[str] = set()
        
    def register_namespace(self, namespace: str):
        """Register a namespace for event organization."""
        self._namespaces.add(namespace)
        
    def on(self, event_pattern: str, priority: int = 0):
        """
        Decorator to register an event handler with pattern matching.
        
        Args:
            event_pattern: Event pattern (supports wildcards: *, **)
            priority: Handler priority (higher runs first)
        """
        def decorator(handler: Callable):
            self.add_handler(event_pattern, handler, priority)
            return handler
        return decorator
    
    def add_handler(self, event_pattern: str, handler: Callable, priority: int = 0):
        """
        Add an event handler with pattern matching support.
        
        Args:
            event_pattern: Event pattern (e.g., "job:*", "**:completed")
            handler: Handler function
            priority: Handler priority
        """
        if event_pattern not in self._handlers:
            self._handlers[event_pattern] = []
        
        self._handlers[event_pattern].append((priority, handler))
        self._handlers[event_pattern].sort(key=lambda x: x[0], reverse=True)
    
    def add_middleware(self, middleware: Callable):
        """
        Add middleware that processes events before handlers.
        
        Args:
            middleware: Function that takes (event, next) and calls next(event)
        """
        self._middleware.append(middleware)
    
    def add_filter(self, filter_func: Callable[[SSEEvent], bool]):
        """
        Add a filter to control which events are processed.
        
        Args:
            filter_func: Function that returns True to process event
        """
        self._filters.append(filter_func)
    
    def add_transformer(self, transformer: Callable[[SSEEvent], SSEEvent]):
        """
        Add a transformer to modify events before processing.
        
        Args:
            transformer: Function that transforms an event
        """
        self._transformers.append(transformer)
    
    def _match_pattern(self, pattern: str, event_type: str) -> bool:
        """
        Check if an event type matches a pattern.
        
        Patterns:
        - exact: "job:created" matches only "job:created"
        - wildcard: "job:*" matches "job:created", "job:updated", etc.
        - deep wildcard: "**:created" matches any namespace with "created"
        - full wildcard: "*" matches everything
        """
        if pattern == "*" or pattern == "**":
            return True
            
        # Convert pattern to regex
        regex_pattern = pattern.replace("**", ".*").replace("*", "[^:]*")
        regex_pattern = f"^{regex_pattern}$"
        
        return bool(re.match(regex_pattern, event_type))
    
    async def dispatch(self, event: Union[SSEEvent, Dict[str, Any]]) -> List[Any]:
        """
        Dispatch an event through the processing pipeline.
        
        Args:
            event: Event to dispatch (SSEEvent or dict)
            
        Returns:
            List of handler results
        """
        # Convert dict to SSEEvent if needed
        if isinstance(event, dict):
            event = SSEEvent(
                type=event.get("type", "unknown"),
                data=event.get("data", {}),
                namespace=event.get("namespace"),
                priority=event.get("priority", 0),
                timestamp=event.get("timestamp")
            )
        
        # Apply filters
        for filter_func in self._filters:
            if not filter_func(event):
                return []  # Event filtered out
        
        # Apply transformers
        for transformer in self._transformers:
            event = transformer(event)
        
        # Process through middleware chain
        async def process_event(evt):
            results = []
            
            # Find matching handlers
            for pattern, handlers in self._handlers.items():
                if self._match_pattern(pattern, evt.full_type):
                    for priority, handler in handlers:
                        if asyncio.iscoroutinefunction(handler):
                            result = await handler(evt)
                        else:
                            result = handler(evt)
                        if result:
                            results.append(result)
            
            return results
        
        # Apply middleware
        current_processor = process_event
        for middleware in reversed(self._middleware):
            next_processor = current_processor
            async def wrapped_processor(evt, next_proc=next_processor):
                if asyncio.iscoroutinefunction(middleware):
                    return await middleware(evt, next_proc)
                else:
                    return middleware(evt, lambda e: asyncio.create_task(next_proc(e)))
            current_processor = wrapped_processor
        
        return await current_processor(event)
    
    def clear_handlers(self, pattern: Optional[str] = None):
        """Clear handlers for a specific pattern or all handlers."""
        if pattern:
            self._handlers.pop(pattern, None)
        else:
            self._handlers.clear()

# Additional UI Helper Functions and Decorators

@dataclass
class SSEMonitorConfig:
    """Configuration for SSE connection monitoring."""
    sse_element_id: str = "sse-connection"
    status_element_id: str = "connection-status"
    auto_reconnect: bool = True
    reconnect_delay: int = 3000
    debug: bool = False
    heartbeat_timeout: int = 30000
    status_indicators: Optional[Dict[str, str]] = None

def create_sse_monitor(config: SSEMonitorConfig):
    """
    Create a connection monitor with the specified configuration.
    
    Args:
        config: SSEMonitorConfig instance
        
    Returns:
        Script element with monitoring code
    """
    return htmx_sse.create_sse_monitor_script({
        'sse_element_id': config.sse_element_id,
        'status_element_id': config.status_element_id,
        'auto_reconnect': config.auto_reconnect,
        'debug': config.debug,
        'status_indicators': config.status_indicators or {
            'active': '<span style="color: green;">●</span> Connected',
            'disconnected': '<span style="color: red;">●</span> Disconnected',
            'error': '<span style="color: orange;">●</span> Error',
            'reconnecting': '<span style="color: blue;">●</span> Reconnecting'
        }
    })

def sse_element(endpoint: str, 
                events: Optional[Union[str, List[str]]] = None,
                auto_close: bool = True,
                swap_type: str = "message"):
    """
    Decorator to add SSE capabilities to any element.
    
    Args:
        endpoint: SSE endpoint URL
        events: Event names to listen for
        auto_close: Whether to auto-close on completion
        swap_type: How to swap content
        
    Returns:
        Decorator function
    """
    def decorator(element_func: Callable):
        def wrapper(*args, **kwargs):
            # Create the base element
            element = element_func(*args, **kwargs)
            
            # Add SSE attributes
            return htmx_sse.add_sse_attrs(
                element,
                endpoint=endpoint,
                events=events,
                swap_type=swap_type,
                auto_reconnect=not auto_close
            )
        return wrapper
    return decorator

# Simplified OOB update function (already exists but making it more prominent)
def oob_update(element_id: str, content: Any, swap_type: str = "innerHTML"):
    """
    Create an out-of-band update element.
    
    Args:
        element_id: Target element ID
        content: Content to swap
        swap_type: Type of swap (innerHTML, outerHTML, etc.)
        
    Returns:
        Element configured for OOB swap
    """
    return oob_element(element_id, content, swap_type)

In [7]:
# Create app
app, rt = create_test_app(theme=DaisyUITheme.BUSINESS)
app.hdrs.append(Link(rel='icon', type='image/svg+xml', href=f'https://api.dicebear.com/9.x/adventurer/svg?seed={random.random()}'))

# Job metadata storage
job_metadata: Dict[str, Any] = {}
job_results: Dict[str, Any] = {}

# Initialize the SSE Broadcast Manager
sse_manager = SSEBroadcastManager(
    max_queue_size=100,
    history_size=50,
    default_timeout=0.1
)

In [8]:
# Initialize SSEElementUpdater, SSEEventDispatcher and register handlers
element_updater = SSEElementUpdater()
event_dispatcher = SSEEventDispatcher()

# Register namespaces for better organization
event_dispatcher.register_namespace("job")
event_dispatcher.register_namespace("queue")
event_dispatcher.register_namespace("ui")

# Add middleware to log all events (optional, for debugging)
@event_dispatcher.add_middleware
async def logging_middleware(event: SSEEvent, next_handler):
    if isinstance(event, SSEEvent):
        print(f"[Event] {event.full_type}: {event.data}")
    return await next_handler(event)

# Register handlers using the event dispatcher with namespace patterns
@event_dispatcher.on("job:created", priority=10)
def dispatch_job_created(event: SSEEvent):
    return element_updater.create_elements("job_created", event.data)

@event_dispatcher.on("job:cancelled", priority=10)
def dispatch_job_cancelled(event: SSEEvent):
    return element_updater.create_elements("job_cancelled", event.data)

@event_dispatcher.on("job:completed", priority=10)
def dispatch_job_completed(event: SSEEvent):
    return element_updater.create_elements("job_completed", event.data)

@event_dispatcher.on("queue:cleared", priority=10)
def dispatch_jobs_cleared(event: SSEEvent):
    return element_updater.create_elements("jobs_cleared", event.data)

@event_dispatcher.on("queue:refresh", priority=10)
def dispatch_queue_refresh(event: SSEEvent):
    return element_updater.create_elements("queue_refresh", event.data)

# Register element update handlers
@element_updater.register("job_created", priority=10)
def handle_job_created(data):
    elements = []
    # Update queue
    elements.append(oob_update(
        "job-queue",
        queue().children[-1] if queue().children else queue()
    ))
    # Disable submit button
    elements.append(render_submit_button(disabled=True, oob_swap=True))
    return elements

@element_updater.register("job_cancelled", priority=10)
def handle_job_cancelled(data):
    elements = []
    job_id = data.get("job_id")
    if job_id:
        snapshot = monitor.snapshot(job_id)
        meta = job_metadata.get(job_id, {})
        if snapshot:
            row = render_job_row(job_id, snapshot, meta)
            elements.append(oob_swap(row, swap_type="outerHTML"))
    
    # Re-enable submit button if no jobs running
    if not has_running_jobs():
        elements.append(render_submit_button(disabled=False, oob_swap=True))
    return elements

@element_updater.register("job_completed", priority=10)
def handle_job_completed(data):
    elements = []
    job_id = data.get("job_id")
    if job_id:
        snapshot = monitor.snapshot(job_id)
        meta = job_metadata.get(job_id, {})
        if snapshot:
            row = render_job_row(job_id, snapshot, meta)
            elements.append(oob_swap(row, swap_type="outerHTML"))
    
    # Re-enable submit button if no jobs running
    if not has_running_jobs():
        elements.append(render_submit_button(disabled=False, oob_swap=True))
    return elements

@element_updater.register("jobs_cleared", priority=10)
def handle_jobs_cleared(data):
    elements = []
    # Refresh entire queue
    queue_content = queue().children[-1] if queue().children else P("No jobs in queue", cls=str(text_color.gray(500)))
    elements.append(oob_update("job-queue", queue_content))
    # Enable submit button
    elements.append(render_submit_button(disabled=False, oob_swap=True))
    return elements

@element_updater.register("queue_refresh", priority=10)
def handle_queue_refresh(data):
    elements = []
    # Full queue refresh
    queue_content = queue().children[-1] if queue().children else queue()
    elements.append(oob_update("job-queue", queue_content))
    # Update button state
    elements.append(render_submit_button(disabled=has_running_jobs(), oob_swap=True))
    return elements

# Add postprocessor to always include stats updates
element_updater.add_postprocessor(lambda elements: elements + render_stats_updates())

# Enhanced broadcast function that uses event dispatcher
async def broadcast_update(update_type: str, data: Dict[str, Any], namespace: str = None):
    """
    Broadcast an update to all connected SSE clients using the event system.
    
    Args:
        update_type: Type of update (created, cancelled, cleared, etc.)
        data: Update data to send
        namespace: Optional namespace for the event
    """
    # Determine namespace from update_type if not provided
    if not namespace:
        if update_type.startswith("job"):
            namespace = "job"
            update_type = update_type.replace("job_", "").replace("job", "")
        elif update_type.startswith("queue"):
            namespace = "queue"
            update_type = update_type.replace("queue_", "").replace("queue", "")
        elif update_type == "jobs_cleared":
            namespace = "queue"
            update_type = "cleared"
    
    # Create SSE event
    event = SSEEvent(
        type=update_type,
        data=data,
        namespace=namespace,
        timestamp=datetime.now().isoformat()
    )
    
    # Broadcast via SSEBroadcastManager
    return await sse_manager.broadcast(event.full_type, data)

# Function to create broadcast elements using the event dispatcher
async def create_broadcast_elements(update_type: str, data: Dict[str, Any]):
    """
    Create HTML elements for broadcast updates using the event dispatcher.
    
    Args:
        update_type: Type of update
        data: Update data
        
    Returns:
        Div containing all OOB swap elements
    """
    # Parse namespace from update_type
    namespace = None
    event_type = update_type
    
    if ":" in update_type:
        namespace, event_type = update_type.split(":", 1)
    elif update_type.startswith("job"):
        namespace = "job"
        event_type = update_type.replace("job_", "").replace("job", "")
    elif update_type.startswith("queue"):
        namespace = "queue"
        event_type = update_type.replace("queue_", "").replace("queue", "")
    elif update_type == "jobs_cleared":
        namespace = "queue"
        event_type = "cleared"
    
    # Create event
    event = SSEEvent(
        type=event_type,
        data=data,
        namespace=namespace,
        timestamp=datetime.now().isoformat()
    )
    
    # Dispatch through event system
    results = await event_dispatcher.dispatch(event)
    
    # Flatten results (handlers return lists of elements)
    elements = []
    for result in results:
        if isinstance(result, list):
            elements.extend(result)
        elif result:
            elements.append(result)
    
    return Div(*elements) if elements else Div()

In [9]:
# Add HTMX SSE extension after HTMX script
def get_htmx_idx(hdrs):
    return next((i for i, hdr in enumerate(hdrs) if (hdr.attrs.get('src') or '').endswith('htmx.min.js')), -1)

htmx_idx = get_htmx_idx(app.hdrs)
if htmx_idx >= 0:
    # Insert SSE extension right after HTMX
    app.hdrs.insert(htmx_idx+1, Script(src="https://unpkg.com/htmx-ext-sse"))
    app.hdrs.insert(htmx_idx+2, Script(code="""window.addEventListener('beforeunload',()=>{document.querySelectorAll('[sse-connect]').forEach(e=>{e['htmx-internal-data']?.sseEventSource?.close()});htmx?.findAll('[sse-connect]').forEach(e=>htmx.trigger(e,'htmx:sseClose'))});"""))
else:
    print("HTMX not found")

In [10]:
# Cancellable job runner extension
class CancellableJobRunner(JobRunner):
    def __init__(self, monitor):
        super().__init__(monitor)
        self._stop_events = {}
    
    def start_cancellable(self, job_id, fn, *args, patch_kwargs=None, **kwargs):
        stop_event = threading.Event()
        self._stop_events[job_id] = stop_event
        
        def wrapper():
            try:
                result = fn(stop_event, *args, **kwargs)
                job_results[job_id] = {"status": "success", "data": result}
            except Exception as e:
                job_results[job_id] = {"status": "error", "error": str(e)}
            finally:
                # Clean up stop event when job finishes
                if job_id in self._stop_events:
                    del self._stop_events[job_id]
        
        return self.start(job_id, wrapper, patch_kwargs=patch_kwargs)
    
    def cancel(self, job_id):
        if job_id in self._stop_events:
            # Set the stop event to signal the job to stop
            self._stop_events[job_id].set()
            
            # Mark job as cancelled in metadata
            if job_id in job_metadata:
                job_metadata[job_id]["status"] = "cancelled"
            
            # Mark job as cancelled in results
            job_results[job_id] = {"status": "cancelled"}
            
            # Force mark as completed in monitor so it can be cleared
            snapshot = monitor.snapshot(job_id)
            if snapshot:
                # Update the monitor's internal state to mark as completed
                monitor._jobs[job_id]['completed'] = True
                monitor._jobs[job_id]['overall_progress'] = snapshot['overall_progress']
            
            return True
        return False

In [11]:
# Initialize with history for job tracking
monitor = ProgressMonitor(keep_history=True, history_limit=200)
# Use cancellable runner
runner = CancellableJobRunner(monitor)

In [12]:
# Helper functions for reducing code duplication

# Job status management
def get_job_status(job_id, job_snapshot=None, metadata=None):
    """
    Determine job status consistently across the application.
    Returns: (status_text, status_color, is_running)
    """
    if metadata is None:
        metadata = job_metadata.get(job_id, {})
    
    if metadata.get('status') == 'cancelled':
        return "Cancelled", badge_colors.error, False
    elif job_snapshot and job_snapshot['completed']:
        return "Complete", badge_colors.success, False
    else:
        return "Running", badge_colors.info, True

def has_running_jobs(exclude_job_id=None):
    """Check if any jobs are currently running (excluding specified job)"""
    all_jobs = monitor.all()
    return any(
        not job['completed'] and 
        job_metadata.get(jid, {}).get('status') != 'cancelled'
        for jid, job in all_jobs.items() 
        if jid != exclude_job_id
    )

def get_submit_button_state(exclude_job_id=None):
    """Determine if submit button should be disabled"""
    return has_running_jobs(exclude_job_id)

# Element ID generation
def job_element_id(element_type, job_id, suffix=""):
    """Generate consistent element IDs for job-related elements"""
    base_id = f"{element_type}-{job_id}"
    return f"{base_id}-{suffix}" if suffix else base_id

# Progress bar rendering helpers
def render_progress_oob(job_id, value, close_sse=True):
    """Render progress bar with optional OOB swap for SSE closing"""
    return Span(
        render_progress_bar(value, color=progress_colors.primary, width=w(20)),
        id=job_element_id("progress-span", job_id),
        hx_swap_oob="true" if close_sse else None
    )

def render_status_oob(job_id, status_text, status_color, close_sse=True):
    """Render status badge with optional OOB swap"""
    return Span(
        render_status_badge(status_text, status_color),
        id=job_element_id("status-span", job_id),
        hx_swap_oob="true" if close_sse else None
    )

def render_actions_oob(job_id, include_cancel=False):
    """Render action buttons with OOB swap"""
    buttons = [render_view_button(job_id)]
    if include_cancel:
        buttons.append(render_cancel_button(job_id))
    
    return Span(
        *buttons,
        id=job_element_id("actions-span", job_id),
        hx_swap_oob="true"
    )

# UI helper functions
def render_submit_button(disabled=False, oob_swap=False):
    """Render submit button with appropriate state"""
    btn_classes = combine_classes(
        btn,
        btn_colors.primary,
        btn_behaviors.disabled if disabled else ""
    )
    
    kwargs = {
        'type': 'submit',
        'id': 'submit-job-btn',
        'cls': btn_classes,
        'disabled': disabled
    }
    
    if oob_swap:
        kwargs['hx_swap_oob'] = 'true'
    
    return Button("Submit Job", **kwargs)

In [13]:
# Job details content building
def build_job_details_content(job_id, snapshot, meta, result, include_sse=False):
    """Build the complete job details content (reusable for static and SSE)"""
    if not snapshot:
        return Div("Job not found", cls=combine_classes(alert, alert_colors.error))
    
    # Build progress bars dynamically
    bars = []
    for bar_id, bar in snapshot['bars'].items():
        bars.append(
            Div(
                P(f"{bar.description}: {bar.progress:.1f}% ({bar.current}/{bar.total})",
                  cls=str(font_size.sm)),
                render_progress_bar(bar.progress, color=progress_colors.accent, width=w.full),
                cls=str(m.b(3))
            )
        )
    
    content = Div(
        # Job info
        Div(
            P(f"ID: {job_id}", cls=combine_classes(font_size.xs, text_color.gray(500))),
            P(f"Name: {meta.get('name', 'Unknown')}", cls=str(font_weight.semibold)),
            P(f"Type: {meta.get('type', 'unknown').title()}"),
            P(f"Created: {meta.get('created_at', 'Unknown')}", cls=str(font_size.sm)),
            cls=str(m.b(4))
        ),
        
        # Overall progress
        Div(
            P(f"Overall Progress: {snapshot['overall_progress']:.1f}%",
              cls=combine_classes(font_weight.bold, m.b(2))),
            render_progress_bar(snapshot['overall_progress'], color=progress_colors.primary, width=w.full),
            cls=str(m.b(4))
        ),
        
        # Individual bars
        Div(
            H4("Task Progress:", cls=combine_classes(font_weight.semibold, m.b(2))),
            *bars
        ) if bars else "",
        
        # Results if available
        Div(
            H4("Results:", cls=combine_classes(font_weight.semibold, m.b(2))),
            Pre(
                json.dumps(result, indent=2),
                cls=combine_classes(
                    bg_dui.base_300, p(3), rounded(),
                    font_size.xs, overflow.auto, max_h(40)
                )
            ),
            cls=str(m.t(4))
        ) if result else "",
        
        # History if available
        Div(
            H4("History:", cls=combine_classes(font_weight.semibold, m.b(2))),
            P(f"{len(snapshot.get('history', []))} updates recorded", cls=str(font_size.sm)),
            cls=str(m.t(4))
        ) if snapshot.get('history') else "",
        
        id="job-details-inner"
    )
    
    # Add SSE connection if requested and job is still running
    if include_sse:
        status_text, _, is_running = get_job_status(job_id, snapshot, meta)
        if is_running:
            # Use HTMXSSEConnector to add SSE attributes
            htmx_sse.add_sse_attrs(
                content,
                endpoint=f"/stream_job_details?job_id={job_id}",
                swap_type="message"
            )
    
    return content

In [14]:
def render_view_button(job_id):
    """Render view button for job details"""
    return Button(
        "View",
        hx_get=f"/job_details?job_id={job_id}",
        hx_target="#job-details-content",
        hx_swap="innerHTML",
        onclick="document.getElementById('job-modal').showModal()",
        cls=combine_classes(btn, btn_sizes.xs, btn_colors.primary)
    )

In [15]:
def render_cancel_button(job_id):
    """Render a cancel button for a job"""
    return Button(
        "Cancel",
        hx_post=f"/cancel_job?job_id={job_id}",
        hx_target="body",
        hx_swap="none",
        cls=combine_classes(btn, btn_sizes.xs, btn_colors.error, m.l(1)),
        id=f"cancel-btn-{job_id}"
    )

In [16]:
def render_progress_bar(value, max_value="100", color=None, width=None):
    """Render a progress bar element"""
    classes = [progress]
    if color:
        classes.append(color)
    if width:
        classes.append(width)
    
    return Progress(
        value=str(value),
        max=max_value,
        cls=combine_classes(*classes)
    )

In [17]:
def render_status_badge(text, color):
    """Render a status badge"""
    return Span(
        text,
        cls=combine_classes(badge, color)
    )

In [18]:
def render_section_header(text):
    """Render a section header (H2)"""
    return H2(text, cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4)))

In [19]:
def render_card_container(*children, extra_classes=None):
    """Render a card container"""
    classes = [card, bg_dui.base_200, p(6), m.b(6)]
    if extra_classes:
        classes.extend(extra_classes if isinstance(extra_classes, list) else [extra_classes])
    
    return Div(*children, cls=combine_classes(*classes))

def render_stats_updates():
    """Render statistics updates as OOB swaps"""
    jobs = monitor.all()
    
    # Calculate current stats
    total = len(jobs)
    running = sum(
        1 for job_id, job in jobs.items()
        if not job['completed'] and job_metadata.get(job_id, {}).get('status') != 'cancelled'
    )
    completed = sum(
        1 for job_id, job in jobs.items()
        if job['completed'] and job_metadata.get(job_id, {}).get('status') != 'cancelled'
    )
    cancelled = sum(
        1 for job_id in job_metadata
        if job_metadata[job_id].get('status') == 'cancelled'
    )
    
    # Return OOB swap elements for all stats
    return [
        Div(str(total), id="stat-total", hx_swap_oob="true", cls=str(stat_value)),
        Div(str(running), id="stat-running", hx_swap_oob="true", cls=combine_classes(stat_value, text_dui.primary)),
        Div(str(completed), id="stat-completed", hx_swap_oob="true", cls=combine_classes(stat_value, text_dui.success)),
        Div(str(cancelled), id="stat-cancelled", hx_swap_oob="true", cls=combine_classes(stat_value, text_dui.error))
    ]

In [20]:
# SSE connection monitoring helpers using HTMXSSEConnector
def create_connection_status_indicators():
    """Create status indicator elements for different connection states"""
    return {
        'active': Span(
            Span(cls=combine_classes(status, status_colors.success, status_sizes.md, m.r(2))),
            Span("Real-time updates active", cls=combine_classes(text_dui.success))
        ),
        'disconnected': Span(
            Span(cls=combine_classes(status, status_colors.error, status_sizes.md, m.r(2))),
            Span("Real-time updates disconnected", cls=combine_classes(text_dui.error))
        ),
        'error': Span(
            Span(cls=combine_classes(status, status_colors.warning, status_sizes.md, m.r(2))),
            Span("Connection error - retrying...", cls=combine_classes(text_dui.warning))
        ),
        'reconnecting': Span(
            Span(cls=combine_classes(status, status_colors.info, status_sizes.md, m.r(2))),
            Span("Reconnecting...", cls=combine_classes(text_dui.info))
        )
    }

def render_sse_connection_monitor(
    sse_element_id="global-sse-connection",
    status_element_id="connection-status",
    auto_reconnect_on_visibility=True,
    debug_logging=True
):
    """
    Create a Script element that monitors SSE connection status using HTMXSSEConnector.
    
    Args:
        sse_element_id: ID of the SSE connection element to monitor
        status_element_id: ID of the element to update with connection status
        auto_reconnect_on_visibility: Whether to reconnect when tab becomes visible
        debug_logging: Whether to log connection events to console
        
    Returns:
        Script element with connection monitoring code
    """
    # Get the status indicators as HTML strings
    indicators = create_connection_status_indicators()
    
    # Convert indicators to HTML strings for JavaScript
    status_indicator_html = {
        'active': str(indicators['active']),
        'disconnected': str(indicators['disconnected']),
        'error': str(indicators['error']),
        'reconnecting': str(indicators['reconnecting'])
    }
    
    # Use HTMXSSEConnector to create the monitoring script
    return htmx_sse.create_sse_monitor_script({
        'sse_element_id': sse_element_id,
        'status_element_id': status_element_id,
        'auto_reconnect': auto_reconnect_on_visibility,
        'debug': debug_logging,
        'status_indicators': status_indicator_html
    })

def render_global_sse_connection(
    endpoint="/stream_global_updates",
    element_id="global-sse-connection",
    hidden=True
):
    """
    Create the global SSE connection element using HTMXSSEConnector.
    
    Args:
        endpoint: The SSE endpoint to connect to
        element_id: ID for the connection element
        hidden: Whether to hide the element
        
    Returns:
        Div element configured for SSE connection
    """
    return htmx_sse.create_sse_element(
        element_type=Div,
        endpoint=endpoint,
        element_id=element_id,
        hidden=hidden
    )

def render_connection_status_display(
    initial_connections=0,
    element_id="connection-status",
    show_count=True
):
    """
    Create the connection status display element.
    
    Args:
        initial_connections: Initial number of connected tabs
        element_id: ID for the status element
        show_count: Whether to show the connection count
        
    Returns:
        Div element for displaying connection status
    """
    indicators = create_connection_status_indicators()
    
    content = [indicators['active']]
    
    if show_count:
        content.append(
            Span(
                f" ({initial_connections} tabs connected)",
                cls=str(font_size.xs)
            )
        )
    
    return Div(
        *content,
        cls=combine_classes(text_dui.success, m.b(4)),
        id=element_id
    )

In [21]:
def render_job_row(job_id, job, meta):
    """Render a complete job row with SSE streaming for active jobs using HTMXSSEConnector"""
    
    # Use centralized status determination
    status_text, status_color, is_running = get_job_status(job_id, job, meta)
    
    if is_running:
        # Active job - use SSE for real-time updates with HTMXSSEConnector
        return Tr(
            Td(job_id[:8] + "...", id=job_element_id("job-id", job_id)),
            Td(meta.get('name', 'Unknown'), id=job_element_id("job-name", job_id)),
            Td(meta.get('type', 'unknown').title(), id=job_element_id("job-type", job_id)),
            Td(
                htmx_sse.sse_progress_element(
                    job_id,
                    initial_content=render_progress_bar(job['overall_progress'], color=progress_colors.primary, width=w(20))
                ),
                id=job_element_id("job-progress", job_id)
            ),
            Td(
                htmx_sse.sse_status_element(
                    job_id,
                    initial_content=render_status_badge(status_text, status_color)
                ),
                id=job_element_id("job-status", job_id)
            ),
            Td(
                Span(
                    render_view_button(job_id),
                    render_cancel_button(job_id),
                    id=job_element_id("actions-span", job_id)
                ),
                id=job_element_id("job-actions", job_id)
            ),
            id=job_element_id("job-row", job_id)
        )
    else:
        # Completed/cancelled job - static display
        return Tr(
            Td(job_id[:8] + "...", id=job_element_id("job-id", job_id)),
            Td(meta.get('name', 'Unknown'), id=job_element_id("job-name", job_id)),
            Td(meta.get('type', 'unknown').title(), id=job_element_id("job-type", job_id)),
            Td(
                render_progress_bar(job['overall_progress'], color=progress_colors.primary, width=w(20)),
                id=job_element_id("job-progress", job_id)
            ),
            Td(
                render_status_badge(status_text, status_color),
                id=job_element_id("job-status", job_id)
            ),
            Td(
                Span(
                    render_view_button(job_id),
                    id=job_element_id("actions-span", job_id)
                ),
                id=job_element_id("job-actions", job_id)
            ),
            id=job_element_id("job-row", job_id)
        )

In [22]:
# Various job types
def batch_processing_job(stop_event, batch_size=1000, delay=0.01):
    from tqdm import tqdm
    import time
    
    results = []
    
    # Process in batches
    for batch in range(3):
        desc = f"Batch {batch + 1}/3"
        for i in tqdm(range(batch_size), desc=desc):
            if stop_event.is_set():
                return {"status": "cancelled", "processed": results}
            time.sleep(delay)
            results.append(f"item_{batch}_{i}")
    
    return {"status": "complete", "processed": results}

In [23]:
def data_export_job(stop_event, format_type="csv", records=200):
    from tqdm import tqdm
    import time
    
    # Fetch data
    for _ in tqdm(range(records // 2), desc="Fetching data"):
        if stop_event.is_set():
            return {"status": "cancelled"}
        time.sleep(0.01)
    
    # Format data
    for _ in tqdm(range(records // 4), desc=f"Formatting as {format_type}"):
        if stop_event.is_set():
            return {"status": "cancelled"}
        time.sleep(0.02)
    
    # Write to file
    for _ in tqdm(range(records // 4), desc="Writing to file"):
        if stop_event.is_set():
            return {"status": "cancelled"}
        time.sleep(0.01)
    
    return {"status": "complete", "file": f"export_{format_type}_{records}.{format_type}"}

In [24]:
def model_training_job(stop_event, epochs=10, batch_size=32):
    from tqdm import tqdm
    import time
    import random
    
    metrics = []
    
    for epoch in range(epochs):
        # Training
        for _ in tqdm(range(1000), desc=f"Epoch {epoch+1}/{epochs} - Training"):
            if stop_event.is_set():
                return {"status": "cancelled", "metrics": metrics}
            time.sleep(0.005)
        
        # Validation
        for _ in tqdm(range(200), desc=f"Epoch {epoch+1}/{epochs} - Validation"):
            if stop_event.is_set():
                return {"status": "cancelled", "metrics": metrics}
            time.sleep(0.01)
        
        # Record metrics
        metrics.append({
            "epoch": epoch + 1,
            "loss": random.uniform(0.1, 1.0),
            "accuracy": random.uniform(0.7, 0.99)
        })
    
    return {"status": "complete", "metrics": metrics}

In [25]:
# Main page with SSE support and cross-tab synchronization
@rt("/")
def index():
    # Configure SSE monitoring
    monitor_config = SSEMonitorConfig(
        sse_element_id="global-sse-connection",
        status_element_id="connection-status",
        auto_reconnect=True,
        debug=True,  # Set to False in production
        status_indicators={
            'active': str(create_connection_status_indicators()['active']),
            'disconnected': str(create_connection_status_indicators()['disconnected']),
            'error': str(create_connection_status_indicators()['error']),
            'reconnecting': str(create_connection_status_indicators()['reconnecting'])
        }
    )
    
    return create_test_page(
        "Job Queue Manager - SSE with Cross-Tab Sync",
        Div(
            H1("Job Queue Management System (SSE + Cross-Tab Sync)", 
               cls=combine_classes(font_size._3xl, font_weight.bold, m.b(6))),
            
            # Global SSE connection for cross-tab synchronization
            render_global_sse_connection(
                endpoint="/stream_global_updates",
                element_id="global-sse-connection",
                hidden=True
            ),
            
            # Connection status indicator
            render_connection_status_display(
                initial_connections=sse_manager.connection_count,
                element_id="connection-status",
                show_count=True
            ),
            
            # Job creation panel
            render_card_container(
                render_section_header("Create New Job"),
                Form(
                    Select(
                        Option("Batch Processing", value="batch"),
                        Option("Data Export", value="export"),
                        Option("Model Training", value="training"),
                        name="type",
                        cls=combine_classes(select, w.full, m.b(3))
                    ),
                    Input(
                        name="name",
                        type="text",
                        placeholder="Job name (optional)",
                        cls=combine_classes(text_input, w.full, m.b(3))
                    ),
                    # Checkbox for allowing concurrent jobs (optional)
                    Label(
                        Input(
                            type="checkbox",
                            name="allow_concurrent",
                            value="true",
                            cls="checkbox checkbox-sm"
                        ),
                        " Allow concurrent jobs (override single job limit)",
                        cls=combine_classes(m.b(3), font_size.sm)
                    ),
                    Div(
                        render_submit_button(disabled=False),
                        Button(
                            "Clear Finished",
                            title="Clear completed and cancelled jobs",
                            hx_post="/clear_completed",
                            hx_target="#job-queue",
                            hx_swap="innerHTML",
                            cls=combine_classes(btn, btn_colors.warning, m.l(2))
                        ),
                        cls=combine_classes(flex_display, gap(2))
                    ),
                    hx_post="/create_job",
                    hx_target="#job-queue",
                    hx_swap="innerHTML",
                    hx_on_after_request="this.reset()",
                    # Client-side protection against double-submit
                    hx_on_before_request="""
                        // Disable submit button immediately on click
                        document.getElementById('submit-job-btn').disabled = true;
                        document.getElementById('submit-job-btn').classList.add('btn-disabled');
                        
                        // Re-enable after response (success or error)
                        htmx.on('htmx:afterRequest', function(evt) {
                            if (evt.detail.elt === evt.currentTarget) {
                                var btn = document.getElementById('submit-job-btn');
                                // Only re-enable if no jobs are running (server will send appropriate state)
                                // The server response will include the correct button state via OOB swap
                            }
                        });
                    """,
                ),
            ),
            
            # Statistics - no SSE connections, will be updated via OOB swaps
            Div(
                render_section_header("Queue Statistics"),
                Div(
                    Div(
                        Div("Total Jobs", cls=str(stat_title)),
                        Div(
                            "0",
                            id="stat-total",
                            cls=str(stat_value)
                        ),
                        cls=str(stat)
                    ),
                    Div(
                        Div("Running", cls=str(stat_title)),
                        Div(
                            "0",
                            id="stat-running",
                            cls=combine_classes(stat_value, text_dui.primary)
                        ),
                        cls=str(stat)
                    ),
                    Div(
                        Div("Completed", cls=str(stat_title)),
                        Div(
                            "0",
                            id="stat-completed",
                            cls=combine_classes(stat_value, text_dui.success)
                        ),
                        cls=str(stat)
                    ),
                    Div(
                        Div("Cancelled", cls=str(stat_title)),
                        Div(
                            "0",
                            id="stat-cancelled",
                            cls=combine_classes(stat_value, text_dui.error)
                        ),
                        cls=str(stat)
                    ),
                    id="queue-stats",
                    cls=combine_classes(stats, shadow(), w.full)
                ),
                cls=str(m.b(6))
            ),
            
            # Job queue table
            render_card_container(
                render_section_header("Job Queue"),
                Div(
                    hx_get="/queue",
                    hx_trigger="load",
                    hx_swap="innerHTML",
                    id="job-queue",
                    cls=str(overflow.x.auto)
                ),
            ),
            
            # Job details modal
            Dialog(
                Div(
                    H3("Job Details", cls=combine_classes(font_weight.bold, font_size.lg, m.b(4))),
                    Div(id="job-details-content"),
                    Div(
                        Button(
                            "Close",
                            onclick="this.closest('dialog').close()",
                            cls=combine_classes(btn, btn_sizes.sm)
                        ),
                        cls=str(modal_action)
                    ),
                    cls=combine_classes(modal_box, max_w._4xl)
                ),
                id="job-modal",
                cls=str(modal)
            ),
            
            # Use the configured SSE monitor
            create_sse_monitor(monitor_config),
            
            cls=combine_classes(container, m.x.auto, p(8))
        )
    )

In [26]:
# API endpoints
@rt("/create_job")
async def create_job(request):
    form = await request.form()
    job_type = form.get('type', 'batch')
    job_name = form.get('name', '')
    allow_concurrent = form.get('allow_concurrent', 'false') == 'true'
    
    # Check if a job is already running (unless concurrent jobs are allowed)
    if not allow_concurrent and has_running_jobs():
        # Return warning message with current queue state
        return Div(
            Div(
                "A job is already running. Please wait for it to complete or cancel it.",
                cls=combine_classes(alert, alert_colors.warning, m.b(4))
            ),
            queue(),
            render_submit_button(disabled=True, oob_swap=True)
        )
    
    job_id = str(uuid.uuid4())
    
    # Select job function with appropriate parameters
    job_configs = {
        'batch': (batch_processing_job, {'batch_size': 50, 'delay': 0.005}),
        'export': (data_export_job, {'format_type': 'csv', 'records': 100}),
        'training': (model_training_job, {'epochs': 3, 'batch_size': 32})
    }
    
    job_fn, kwargs = job_configs.get(job_type, (batch_processing_job, {}))
    
    # Store metadata
    job_metadata[job_id] = {
        'id': job_id,
        'name': job_name or f"{job_type.title()} Job",
        'type': job_type,
        'status': 'running',
        'created_at': datetime.now().isoformat(),
        'params': kwargs
    }
    
    # Start job with appropriate throttling
    runner.start_cancellable(
        job_id,
        job_fn,
        **kwargs,
        patch_kwargs={
            "min_delta_pct": 5,
            "min_update_interval": 0.05,
            "emit_initial": True
        }
    )
    
    # Small delay to allow monitor to register the job
    await asyncio.sleep(0.1)
    
    # Broadcast the job creation to all connected clients with explicit namespace
    await broadcast_update("created", {
        "job_id": job_id,
        "name": job_metadata[job_id]['name'],
        "type": job_type
    }, namespace="job")
    
    # Return response for the initiating client
    # The broadcast will update other tabs
    queue_content = queue()
    button_update = render_submit_button(disabled=True, oob_swap=True)
    stats_updates = render_stats_updates()
    
    return Div(
        queue_content,
        button_update,
        *stats_updates
    )

In [27]:
@rt("/queue")
def queue():
    jobs = monitor.all()
    
    if not jobs:
        button_update = render_submit_button(disabled=False, oob_swap=True)
        stats_updates = render_stats_updates()
        result = P("No jobs in queue", cls=str(text_color.gray(500)))
        return Div(button_update, *stats_updates, result)
    
    # Use centralized check for running jobs
    has_running = has_running_jobs()
    
    rows = []
    for job_id, job in jobs.items():
        meta = job_metadata.get(job_id, {})
        rows.append(render_job_row(job_id, job, meta))
    
    # Update button state using helper
    button_update = render_submit_button(disabled=has_running, oob_swap=True)
    
    # Get stats updates
    stats_updates = render_stats_updates()
    
    # Build the table
    table_content = Table(
        Thead(
            Tr(
                Th("ID"),
                Th("Name"),
                Th("Type"),
                Th("Progress"),
                Th("Status"),
                Th("Actions")
            )
        ),
        Tbody(*rows),
        cls=combine_classes(table, table_modifiers.zebra, w.full)
    )
    
    return Div(button_update, *stats_updates, table_content)

In [28]:
# SSE streaming endpoints for job progress
@rt("/stream_job_progress")
def stream_job_progress(job_id: str):
    """SSE endpoint for streaming job progress bar updates"""
    
    async def progress_stream():
        try:
            # Check if job exists before starting stream
            snapshot = monitor.snapshot(job_id)
            meta = job_metadata.get(job_id, {})
            status_text, _, is_running = get_job_status(job_id, snapshot, meta)
            
            if not snapshot or not is_running:
                # Send final static progress bar
                progress_value = snapshot['overall_progress'] if snapshot else 0
                yield sse_message(render_progress_oob(job_id, progress_value, close_sse=True))
                return
            
            async for data in sse_stream_async(
                monitor,
                job_id,
                interval=0.5,
                heartbeat=30.0,
                wait_for_start=False,
                start_timeout=5.0
            ):
                if data.startswith('data: '):
                    try:
                        json_str = data[6:].strip()
                        if json_str and json_str != '{}':
                            progress_data = json.loads(json_str)
                            
                            # Check if job was cancelled
                            meta = job_metadata.get(job_id, {})
                            if meta.get('status') == 'cancelled':
                                # Send final progress and close SSE
                                yield sse_message(render_progress_oob(
                                    job_id, 
                                    progress_data.get('progress', 0), 
                                    close_sse=True
                                ))
                                break
                            
                            if progress_data.get('completed'):
                                # Send final progress bar with OOB swap to close SSE
                                yield sse_message(render_progress_oob(job_id, 100, close_sse=True))
                                break
                            else:
                                # Send progress update
                                progress_value = progress_data.get('progress', 0)
                                yield sse_message(
                                    render_progress_bar(progress_value, color=progress_colors.primary, width=w(20))
                                )
                    except json.JSONDecodeError:
                        pass
                elif data.startswith('event: end'):
                    # Job not found or error
                    yield sse_message(render_progress_oob(job_id, 0, close_sse=True))
                    break
                elif data.startswith(': '):
                    yield data  # Heartbeat
        except Exception as e:
            print(f"Error in progress stream for {job_id}: {e}")
            yield sse_message(render_progress_oob(job_id, 0, close_sse=True))
    
    return EventStream(progress_stream())

In [29]:
@rt("/stream_job_status")
def stream_job_status(job_id: str):
    """SSE endpoint for streaming job status updates"""
    
    async def status_stream():
        try:
            # Check initial status
            snapshot = monitor.snapshot(job_id)
            meta = job_metadata.get(job_id, {})
            status_text, status_color, is_running = get_job_status(job_id, snapshot, meta)
            
            if not snapshot or not is_running:
                # Send final status
                stats_updates = render_stats_updates()
                yield sse_message(
                    Div(
                        render_status_oob(job_id, status_text, status_color, close_sse=True),
                        render_actions_oob(job_id, include_cancel=False),
                        *stats_updates
                    )
                )
                return
            
            async for data in sse_stream_async(
                monitor,
                job_id,
                interval=0.5,
                heartbeat=30.0,
                wait_for_start=False,
                start_timeout=5.0
            ):
                if data.startswith('data: '):
                    try:
                        json_str = data[6:].strip()
                        if json_str and json_str != '{}':
                            progress_data = json.loads(json_str)
                            
                            # Check if job was cancelled
                            meta = job_metadata.get(job_id, {})
                            if meta.get('status') == 'cancelled':
                                # Check if any other jobs are running for submit button state
                                button_disabled = has_running_jobs(exclude_job_id=job_id)
                                stats_updates = render_stats_updates()
                                
                                # Build the list of OOB elements
                                oob_elements = [
                                    render_status_oob(job_id, "Cancelled", badge_colors.error, close_sse=True),
                                    render_actions_oob(job_id, include_cancel=False),
                                    *stats_updates
                                ]
                                
                                # Add submit button update if no other jobs are running
                                if not button_disabled:
                                    oob_elements.append(render_submit_button(disabled=False, oob_swap=True))
                                
                                yield sse_message(Div(*oob_elements))
                                break
                            
                            if progress_data.get('completed'):
                                # Broadcast job completion to all clients with explicit namespace
                                await broadcast_update("completed", {
                                    "job_id": job_id,
                                    "name": meta.get('name', 'Unknown')
                                }, namespace="job")
                                
                                # Check if any other jobs are running
                                button_disabled = has_running_jobs(exclude_job_id=job_id)
                                stats_updates = render_stats_updates()
                                
                                # Build the list of OOB elements
                                oob_elements = [
                                    render_status_oob(job_id, "Complete", badge_colors.success, close_sse=True),
                                    render_actions_oob(job_id, include_cancel=False),
                                    *stats_updates
                                ]
                                
                                # Add submit button update if no other jobs are running
                                if not button_disabled:
                                    oob_elements.append(render_submit_button(disabled=False, oob_swap=True))
                                
                                yield sse_message(Div(*oob_elements))
                                break
                            else:
                                # Still running - send status update
                                yield sse_message(
                                    render_status_badge("Running", badge_colors.info)
                                )
                    except json.JSONDecodeError:
                        pass
                elif data.startswith('event: end'):
                    yield sse_message(render_status_oob(job_id, "Error", badge_colors.error, close_sse=True))
                    break
                elif data.startswith(': '):
                    yield data  # Heartbeat
        except Exception as e:
            print(f"Error in status stream for {job_id}: {e}")
            yield sse_message(render_status_oob(job_id, "Error", badge_colors.error, close_sse=True))
    
    return EventStream(status_stream())

In [30]:
@rt("/cancel_job")
async def cancel_job(job_id: str):
    """Cancel a job and broadcast the update to all connected clients"""
    success = runner.cancel(job_id)
    
    if success:
        # Broadcast the cancellation to all connected clients with explicit namespace
        await broadcast_update("cancelled", {
            "job_id": job_id,
            "name": job_metadata.get(job_id, {}).get('name', 'Unknown')
        }, namespace="job")
        
        # Return empty response - the broadcast handles UI updates
        return ""
    else:
        print(f"[DEBUG cancel_job] Failed to cancel job {job_id[:8]}")
        return Div(
            "Failed to cancel job",
            cls=combine_classes(alert, alert_colors.error),
            hx_swap_oob="true"
        )

In [31]:
@rt("/job_details")
def job_details(job_id: str):
    """Job details with SSE auto-refresh while running"""
    snapshot = monitor.snapshot(job_id)
    meta = job_metadata.get(job_id, {})
    result = job_results.get(job_id)
    
    # Use the new centralized helper with SSE support
    return build_job_details_content(job_id, snapshot, meta, result, include_sse=True)

In [32]:
@rt("/stream_job_details")
def stream_job_details(job_id: str):
    """SSE endpoint for streaming job details updates"""
    
    async def details_stream():
        try:
            while True:
                snapshot = monitor.snapshot(job_id)
                meta = job_metadata.get(job_id, {})
                status_text, _, is_running = get_job_status(job_id, snapshot, meta)
                
                if not snapshot or not is_running:
                    # Send final update and close
                    result = job_results.get(job_id)
                    
                    # Build final content using the helper, but with OOB swap
                    final_content = build_job_details_content(job_id, snapshot, meta, result, include_sse=False)
                    final_content.attrs['hx-swap-oob'] = 'true'
                    
                    yield sse_message(final_content)
                    break
                
                # Send update using the helper
                yield sse_message(build_job_details_content(job_id, snapshot, meta, job_results.get(job_id), include_sse=False))
                await asyncio.sleep(0.5)
                
        except Exception as e:
            print(f"Error in job details stream for {job_id}: {e}")
    
    return EventStream(details_stream())

In [33]:
@rt("/clear_completed")
async def clear_completed():
    """Clear completed and cancelled jobs, broadcast to all clients"""
    # Get all jobs before clearing
    all_jobs = monitor.all()
    cleared_jobs = []
    
    # Clear completed jobs from monitor
    monitor.clear_completed(older_than_seconds=0)
    
    # Also manually remove cancelled jobs
    for job_id in list(all_jobs.keys()):
        meta = job_metadata.get(job_id, {})
        job = all_jobs.get(job_id, {})
        
        # Use helper to determine status
        status_text, _, is_running = get_job_status(job_id, job, meta)
        
        # Remove if not running (completed or cancelled)
        if not is_running:
            cleared_jobs.append({
                "job_id": job_id,
                "name": meta.get('name', 'Unknown')
            })
            
            # Remove from monitor if still there
            if job_id in monitor._jobs:
                del monitor._jobs[job_id]
            
            # Clean up metadata
            if job_id in job_metadata:
                del job_metadata[job_id]
            
            # Clean up results
            if job_id in job_results:
                del job_results[job_id]
    
    # Broadcast the clearing to all connected clients with explicit namespace
    await broadcast_update("cleared", {
        "cleared_count": len(cleared_jobs),
        "cleared_jobs": cleared_jobs
    }, namespace="queue")
    
    # Return updated queue for the initiating client
    queue_content = queue()
    stats_updates = render_stats_updates()
    
    return Div(
        queue_content,
        *stats_updates
    )

In [34]:
# Global SSE endpoint for cross-tab synchronization
@rt("/stream_global_updates")
async def stream_global_updates():
    """
    SSE endpoint for broadcasting updates to all connected clients.
    This enables cross-tab synchronization of job queue state.
    """
    async def global_stream():
        # Register this connection with SSEBroadcastManager
        queue = await sse_manager.register_connection()
        
        try:
            # Send initial connection confirmation
            yield f": Connected to global updates (active connections: {sse_manager.connection_count})\n\n"
            
            # Send heartbeat and updates
            while True:
                try:
                    # Wait for message with timeout for heartbeat
                    message = await asyncio.wait_for(queue.get(), timeout=30.0)
                    
                    # Create OOB elements based on message type using the event dispatcher
                    elements = await create_broadcast_elements(message["type"], message.get("data", {}))
                    
                    # Send as SSE message
                    yield sse_message(elements)
                    
                except asyncio.TimeoutError:
                    # Send heartbeat to keep connection alive
                    yield f": heartbeat {datetime.now().isoformat()}\n\n"
                    
                except Exception as e:
                    print(f"Error in global stream: {e}")
                    break
                    
        finally:
            # Unregister this connection
            await sse_manager.unregister_connection(queue)
    
    return EventStream(global_stream())

In [35]:
# Start server for Jupyter
server = start_test_server(app, port=8000)
display(HTMX(port=server.port))

In [36]:
# Stop server when done
server.stop()